In [3]:
import pandas as pd
from tqdm import tqdm
import sys
from pathlib import Path

sys.path.append(str(Path("chandas-detector").resolve()))
import chandas_detector
from chandas_detector import detect_meter, format_result

from sentence_transformers import SentenceTransformer
from skrutable.meter_identification import MeterIdentifier

In [4]:
FILEPATH = "Outputs/3_7_high_single_verse_gemini_results.csv"

df = pd.read_csv(FILEPATH)

df.head()


,translation,ground_truth,generated,is_flawed
0,"Vanaras went adoring the Ankola, Karanja, Pla...",अङ्कोलांश्च करञ्चांश्च प्लक्षन्यग्रोधतिन्दुका...,<verse>\nअङ्कोलप्लक्षजम्बूश्च न्यग्रोधं च करञ्...,False
1,"Rama, the best among men and true to his prom...",तासां रामस्समुत्थाय जग्राह चरणान् शुभान्।मात्...,<verse>\nसत्यसन्धो नरश्रेष्ठ उत्थाय रघुनन्दनः ...,False
2,"Having lost all their relations, the demoness...",अद्य विप्रसरिष्यन्ति राक्षस्यो हतबान्धवाः।बाष...,<verse>\nजनयन्त्यः परां भीतिं राक्षस्यो हतबान्...,False
3,I could not even perform the last rites of th...,किं नु तस्य मया कार्यं दुर्जातेन महात्मनः।यो ...,<verse>\nमन्निमित्तेन शोकार्तो यः प्रेतोऽभून्म...,False
4,Heroic Hanuman felt happy when he saw Sita. H...,नमस्कृत्वा च रामाय लक्ष्मणाय च वीर्यवान्।सीता...,<verse>\nदृष्ट्वा सीतां महावीरो नत्वा रामं सलक...,False


In [5]:
print(df['generated'][0])

<verse>
अङ्कोलप्लक्षजम्बूश्च न्यग्रोधं च करञ्जकम् ।
नीपमामलकं चैव स्तुवन्तो वानरा ययुः ॥
</verse>


In [6]:
df['generated'] = df['generated'].apply(lambda x: x.replace("<verse>", "").replace("</verse>", "").strip())

print(df['generated'][0])

अङ्कोलप्लक्षजम्बूश्च न्यग्रोधं च करञ्जकम् ।
नीपमामलकं चैव स्तुवन्तो वानरा ययुः ॥


# Get meter predictions from chandas detector (ours)

In [7]:
import sys
from pathlib import Path

sys.path.append(str(Path("chandas-detector").resolve()))

import chandas_detector
from chandas_detector import detect_meter, format_result

In [8]:
meter_codes = []

for r in tqdm(df.itertuples(index=False), total=len(df)):
    verse = r.generated.strip()
    result = detect_meter(verse)

    meter_codes.append(result.meter if result.confidence == "exact" else None)

df["meter_cd"] = meter_codes

100%|██████████| 1421/1421 [00:00<00:00, 5673.48it/s]


In [9]:
df.head()

,translation,ground_truth,generated,is_flawed,meter_cd
0,"Vanaras went adoring the Ankola, Karanja, Pla...",अङ्कोलांश्च करञ्चांश्च प्लक्षन्यग्रोधतिन्दुका...,अङ्कोलप्लक्षजम्बूश्च न्यग्रोधं च करञ्जकम् ।\nन...,False,Anuṣṭubh
1,"Rama, the best among men and true to his prom...",तासां रामस्समुत्थाय जग्राह चरणान् शुभान्।मात्...,सत्यसन्धो नरश्रेष्ठ उत्थाय रघुनन्दनः ।\nसर्वास...,False,Anuṣṭubh
2,"Having lost all their relations, the demoness...",अद्य विप्रसरिष्यन्ति राक्षस्यो हतबान्धवाः।बाष...,जनयन्त्यः परां भीतिं राक्षस्यो हतबान्धवाः।\nबा...,False,Anuṣṭubh
3,I could not even perform the last rites of th...,किं नु तस्य मया कार्यं दुर्जातेन महात्मनः।यो ...,मन्निमित्तेन शोकार्तो यः प्रेतोऽभून्महामतिः ।\...,False,Anuṣṭubh
4,Heroic Hanuman felt happy when he saw Sita. H...,नमस्कृत्वा च रामाय लक्ष्मणाय च वीर्यवान्।सीता...,दृष्ट्वा सीतां महावीरो नत्वा रामं सलक्ष्मणम् ।...,False,Anuṣṭubh


# Get Syllable counts using skrutable

In [10]:
from skrutable.meter_identification import MeterIdentifier, flush_profiling_report

MI = MeterIdentifier()

In [11]:
syllable_counts = []

for r in tqdm(df.itertuples(index=False), total=len(df)):
    verse = r.generated.strip()
    meter_info = MI.identify_meter(verse, resplit_option="resplit_lite", resplit_keep_midpoint=True, from_scheme="DEV")
    syllables = [x for x in meter_info.text_syllabified.split() if x.strip()]
    syllable_counts.append(len(syllables))


df["syllable_count"] = syllable_counts

100%|██████████| 1421/1421 [00:00<00:00, 1773.05it/s]


# ADD FULL AND HALF SCORES

In [12]:
df['meter_cd'].unique()

<ArrowStringArray>
['Anuṣṭubh', nan]
Length: 2, dtype: str

In [13]:
full = []
half = []

for r in tqdm(df.itertuples(), total=len(df)):
    meter = r.meter_cd
    sc = r.syllable_count
    
    if meter=='Anuṣṭubh' and sc==32:
        full.append(1)
        half.append(1)
    elif meter=='Anuṣṭubh' and sc!=32:
        print("Impossible sample at index: ", r.Index)
        full.append(0)
        half.append(0)
    elif meter!='Anuṣṭubh' and sc==32:
        full.append(0)
        half.append(1)
    else:
        full.append(0)
        half.append(0)
        
df["full"] = full
df["half"] = half

100%|██████████| 1421/1421 [00:00<00:00, 207799.53it/s]


# Get semantic similarity

In [14]:
from sentence_transformers import SentenceTransformer
semantic_model = SentenceTransformer('sanganaka/bge-m3-sanskritFT')

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 4218.40it/s]


In [15]:
from torch import pi, acos

In [16]:
sim_df = df.dropna(subset=["translation", "generated"]).copy()

inputs = sim_df["translation"].astype(str).tolist()
poetry_outputs = sim_df["generated"].astype(str).tolist()

in_embs = semantic_model.encode(inputs, convert_to_tensor=True)
out_embs = semantic_model.encode(poetry_outputs, convert_to_tensor=True)

# Raw cosine similarity — keep as Tensor
sims = semantic_model.similarity_pairwise(in_embs, out_embs)

# Store raw cosine similarity
df["semsim"] = pd.NA
df.loc[sim_df.index, "semsim"] = sims.cpu().numpy()

# Convert cosine similarity to 0–100%
sem_sim_pct = (
    (pi - acos(sims)) * 100 / pi
)

# Store percentage
df["semantic_sim_pct"] = pd.NA
df.loc[sim_df.index, "semantic_sim_pct"] = sem_sim_pct.cpu().numpy()

# Get Results

In [17]:
df.head()

,translation,ground_truth,generated,is_flawed,meter_cd,syllable_count,full,half,semsim,semantic_sim_pct
0,"Vanaras went adoring the Ankola, Karanja, Pla...",अङ्कोलांश्च करञ्चांश्च प्लक्षन्यग्रोधतिन्दुका...,अङ्कोलप्लक्षजम्बूश्च न्यग्रोधं च करञ्जकम् ।\nन...,False,Anuṣṭubh,32,1,1,0.754239,77.199387
1,"Rama, the best among men and true to his prom...",तासां रामस्समुत्थाय जग्राह चरणान् शुभान्।मात्...,सत्यसन्धो नरश्रेष्ठ उत्थाय रघुनन्दनः ।\nसर्वास...,False,Anuṣṭubh,32,1,1,0.78378,78.67112
2,"Having lost all their relations, the demoness...",अद्य विप्रसरिष्यन्ति राक्षस्यो हतबान्धवाः।बाष...,जनयन्त्यः परां भीतिं राक्षस्यो हतबान्धवाः।\nबा...,False,Anuṣṭubh,32,1,1,0.608124,70.807747
3,I could not even perform the last rites of th...,किं नु तस्य मया कार्यं दुर्जातेन महात्मनः।यो ...,मन्निमित्तेन शोकार्तो यः प्रेतोऽभून्महामतिः ।\...,False,Anuṣṭubh,32,1,1,0.74847,76.921112
4,Heroic Hanuman felt happy when he saw Sita. H...,नमस्कृत्वा च रामाय लक्ष्मणाय च वीर्यवान्।सीता...,दृष्ट्वा सीतां महावीरो नत्वा रामं सलक्ष्मणम् ।...,False,Anuṣṭubh,32,1,1,0.658747,72.891335


In [18]:
print("Full%:", df["full"].mean()*100)
print("Half%:", df["half"].mean()*100)
print("SemSim%:", df["semantic_sim_pct"].mean())

Full%: 97.25545390570021
Half%: 99.08515130190007
SemSim%: 76.12829878160771
